Judul: Klasifikasi kain defektif dan kain non defektif menggunakan fitur GLCM dengan membandingkan teknik preprocessing.
Percobaan 2 (Resize + Grayscale + Median Filter + Histogram Equalization)
Nama Anggota: Deswita Salsabila (F1D02410004)
Nama Anggota: Winona Andien Jihan Habbibah (F1D02410027)
Nama Anggota: Faiz Ahmad Tsaqib Wirawan (F1D02410043)
Nama Anggota: Dita Ayu Julita (F1D02410111)

Pada percobaan kedua, preprocessing yang digunakan adalah Resize, Grayscale, Median Filter, dan Histogram Equalization. Median Filter digunakan untuk mengurangi noise pada citra tanpa menghilangkan tepi penting, sedangkan Histogram Equalization digunakan untuk meningkatkan kontras citra agar perbedaan tekstur antara area cacat dan area normal menjadi lebih jelas. Tujuannya adalah melihat apakah penambahan kedua proses ini dapat meningkatkan performa klasifikasi dibandingkan dengan percobaan pertama yang hanya menggunakan Resize dan Grayscale.

In [ ]:
import os
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay, precision_score, recall_score, f1_score
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import seaborn as sns

DATA LOADING

In [ ]:
data = []
labels = []
file_name = []

TARGET_SIZE = (128, 128)
DATASET_PATH = "dataset"

for sub_folder in os.listdir(DATASET_PATH):
    sub_folder_path = os.path.join(DATASET_PATH, sub_folder)
    if not os.path.isdir(sub_folder_path):
        continue
    for filename in os.listdir(sub_folder_path):
        img_path = os.path.join(sub_folder_path, filename)
        img = cv.imread(img_path)
        if img is None:
            continue
        img = img.astype(np.uint8)
        img = cv.resize(img, TARGET_SIZE)
        img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        data.append(img)
        labels.append(sub_folder)
        file_name.append(filename)

data = np.array(data)
labels = np.array(labels)
print(f"Total data: {len(data)}")
print(f"Label unik: {np.unique(labels)}")

DATA UNDERSTANDING

In [ ]:
unique, counts = np.unique(labels, return_counts=True)
print("Distribusi data:")
for u, c in zip(unique, counts):
    print(f"  {u}: {c} gambar")

plt.figure(figsize=(6, 4))
plt.bar(unique, counts, color=['#e74c3c', '#2ecc71'])
plt.title('Distribusi Data per Kelas')
plt.xlabel('Kelas')
plt.ylabel('Jumlah Gambar')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for idx, label in enumerate(unique):
    indices = np.where(labels == label)[0]
    for j in range(5):
        axes[idx][j].imshow(data[indices[j]], cmap='gray')
        axes[idx][j].set_title(f'{label}')
        axes[idx][j].axis('off')
plt.suptitle('Sampel Data per Kelas')
plt.tight_layout()
plt.show()

Dataset yang digunakan terdiri dari 170 citra kain, dengan 85 citra kelas defect dan 85 citra kelas non_defect. Jumlah data antar kelas sudah seimbang sehingga tidak diperlukan augmentasi data.

Terlihat bahwa kelas defect memiliki area cacat berupa bintik atau bercak gelap yang kontras dengan tekstur kain di sekitarnya, serta beberapa citra menunjukkan garis horizontal yang memperlihatkan kerusakan pada serat kain. Sementara itu, kelas non_defect menunjukkan tekstur kain yang lebih bagus tanpa adanya bercak.

DATA AUGMENTATION

In [ ]:
print(f"Total data: {len(data)}")
print("Augmentasi tidak diperlukan karena jumlah data sudah mencukupi.")

DATA PREPARATION & DEFINE PREPROCESSING FUNCTION

In [ ]:
def resize(image, target_size):
    return cv.resize(image, target_size)

def prepro2(image):
    """
    Percobaan 2
    Preprocessing: Resize + Grayscale + Median Filter + Histogram Equalization
    Resize dan Grayscale sudah dilakukan saat load data.
    Pada fungsi ini dilakukan Median Filter untuk mengurangi noise,
    kemudian Histogram Equalization untuk meningkatkan kontras citra.
    """
    image = image.astype(np.uint8)
    image = cv.medianBlur(image, 3)
    image = cv.equalizeHist(image)
    return image

PREPROCESSING

In [ ]:
dataPreprocessed = []
for i in range(len(data)):
    processed = prepro2(data[i])
    dataPreprocessed.append(processed)

dataPreprocessed = np.array(dataPreprocessed)
print(f"Shape data setelah preprocessing: {dataPreprocessed.shape}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for idx, label in enumerate(unique):
    indices = np.where(labels == label)[0]
    for j in range(5):
        axes[idx][j].imshow(dataPreprocessed[indices[j]], cmap='gray')
        axes[idx][j].set_title(f'{label}')
        axes[idx][j].axis('off')
plt.suptitle('Hasil Preprocessing - Percobaan 2 (Resize + Grayscale + Median Filter + Histogram Equalization)')
plt.tight_layout()
plt.show()

In [ ]:
idx_contoh = 0

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0][0].imshow(data[idx_contoh], cmap='gray')
axes[0][0].set_title('Sebelum Preprocessing')
axes[0][0].axis('off')

axes[0][1].imshow(dataPreprocessed[idx_contoh], cmap='gray')
axes[0][1].set_title('Sesudah Median Filter + Histogram Equalization')
axes[0][1].axis('off')

axes[1][0].hist(data[idx_contoh].ravel(), bins=256, range=(0, 255), color='steelblue')
axes[1][0].set_title('Histogram Sebelum')
axes[1][0].set_xlabel('Intensitas Piksel')
axes[1][0].set_ylabel('Frekuensi')

axes[1][1].hist(dataPreprocessed[idx_contoh].ravel(), bins=256, range=(0, 255), color='darkorange')
axes[1][1].set_title('Histogram Sesudah')
axes[1][1].set_xlabel('Intensitas Piksel')
axes[1][1].set_ylabel('Frekuensi')

plt.suptitle('Perbandingan Citra dan Histogram Sebelum & Sesudah Preprocessing')
plt.tight_layout()
plt.show()

FEATURE EXTRACTION

In [ ]:
def glcm(image, derajat, distance=1):
    if derajat == 0:
        angles = [0]
    elif derajat == 45:
        angles = [np.pi / 4]
    elif derajat == 90:
        angles = [np.pi / 2]
    elif derajat == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Sudut harus salah satu dari: 0, 45, 90, 135")
    matriks = graycomatrix(image, [distance], angles, 256, symmetric=True, normed=True)
    return matriks

In [ ]:
def get_contrast(matriks):
    return graycoprops(matriks, 'contrast')[0, 0]

def get_dissimilarity(matriks):
    return graycoprops(matriks, 'dissimilarity')[0, 0]

def get_homogeneity(matriks):
    return graycoprops(matriks, 'homogeneity')[0, 0]

def get_energy(matriks):
    return graycoprops(matriks, 'energy')[0, 0]

def get_correlation(matriks):
    return graycoprops(matriks, 'correlation')[0, 0]

def get_ASM(matriks):
    return graycoprops(matriks, 'ASM')[0, 0]

def get_entropy(matriks):
    return entropy(matriks.ravel())

In [ ]:
DISTANCES = [1, 2, 3, 4, 5]
ANGLES = [0, 45, 90, 135]

all_features = []

for i in range(len(dataPreprocessed)):
    img_features = {}
    for d in DISTANCES:
        for angle in ANGLES:
            mat = glcm(dataPreprocessed[i], angle, distance=d)
            img_features[f'Contrast_d{d}_a{angle}']     = get_contrast(mat)
            img_features[f'Dissimilarity_d{d}_a{angle}'] = get_dissimilarity(mat)
            img_features[f'Homogeneity_d{d}_a{angle}']  = get_homogeneity(mat)
            img_features[f'Energy_d{d}_a{angle}']       = get_energy(mat)
            img_features[f'Correlation_d{d}_a{angle}']  = get_correlation(mat)
            img_features[f'ASM_d{d}_a{angle}']          = get_ASM(mat)
            img_features[f'Entropy_d{d}_a{angle}']      = get_entropy(mat)
    all_features.append(img_features)

print(f"Jumlah fitur per gambar: {len(all_features[0])}")

In [ ]:
dataTable = pd.DataFrame(all_features)
dataTable.insert(0, 'Filename', file_name)
dataTable.insert(1, 'Label', labels)
dataTable.to_csv('hasil_ekstraksi_2.csv', index=False)

hasilEkstrak = pd.read_csv('hasil_ekstraksi_2.csv')
hasilEkstrak.head()

FEATURES SELECTION

In [ ]:
correlation_matrix = hasilEkstrak.drop(columns=['Label', 'Filename']).corr()

threshold = 0.95
columns = np.full((correlation_matrix.shape[0],), True, dtype=bool)
for i in range(correlation_matrix.shape[0]):
    for j in range(i + 1, correlation_matrix.shape[0]):
        if abs(correlation_matrix.iloc[i, j]) >= threshold:
            if columns[j]:
                columns[j] = False

select = hasilEkstrak.drop(columns=['Label', 'Filename']).columns[columns]
x_new = hasilEkstrak[select]
y = hasilEkstrak['Label']

print(f"Jumlah fitur sebelum seleksi: {len(hasilEkstrak.columns) - 2}")
print(f"Jumlah fitur setelah seleksi: {len(select)}")

plt.figure(figsize=(17, 17))
sns.heatmap(x_new.corr(), annot=False, cmap='Blues')
plt.title('Heatmap Korelasi Fitur Setelah Seleksi')
plt.tight_layout()
plt.show()

SPLITTING DATA

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x_new, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

FEATURE NORMALIZATION

In [ ]:
train_mean = X_train.mean()
train_std  = X_train.std()

X_train = (X_train - train_mean) / train_std
X_test  = (X_test  - train_mean) / train_std

print("Normalisasi selesai.")

MODELING & DEFINE MODEL

In [ ]:
def generateClassificationReport(y_true, y_pred):
    print(classification_report(y_true, y_pred))
    print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'Recall   : {recall_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'F1-Score : {f1_score(y_true, y_pred, average="weighted"):.4f}')

rf  = RandomForestClassifier(n_estimators=5, random_state=42)
svm = SVC(kernel='rbf', random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

TRAIN RANDOM FOREST CLASSIFIER

In [ ]:
rf.fit(X_train, y_train)

print("------Training Set------")
generateClassificationReport(y_train, rf.predict(X_train))

print("\n------Testing Set------")
generateClassificationReport(y_test, rf.predict(X_test))

TRAIN SVM CLASSIFIER

In [ ]:
svm.fit(X_train, y_train)

print("------Training Set------")
generateClassificationReport(y_train, svm.predict(X_train))

print("\n------Testing Set------")
generateClassificationReport(y_test, svm.predict(X_test))

TRAIN KNN CLASSIFIER

In [ ]:
knn.fit(X_train, y_train)

print("------Training Set------")
generateClassificationReport(y_train, knn.predict(X_train))

print("\n------Testing Set------")
generateClassificationReport(y_test, knn.predict(X_test))

EVALUATION WITH CONFUSION MATRIX

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_true))
    disp.plot(cmap=plt.cm.Blues)
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_test, rf.predict(X_test),  "Random Forest — Percobaan 2")
plot_confusion_matrix(y_test, svm.predict(X_test), "SVM — Percobaan 2")
plot_confusion_matrix(y_test, knn.predict(X_test), "KNN — Percobaan 2")

Pada percobaan 2, preprocessing tambahan berupa Median Filter dan Histogram Equalization diterapkan setelah Resize dan Grayscale. Median Filter berhasil mengurangi noise pada citra tanpa menghilangkan tekstur penting, sementara Histogram Equalization meningkatkan kontras sehingga perbedaan antara area cacat dan area normal pada kain menjadi lebih terlihat baik secara visual maupun pada distribusi histogramnya.

Berdasarkan hasil klasifikasi, perlu dibandingkan apakah ketiga model (Random Forest, SVM, dan KNN) mengalami peningkatan akurasi, precision, recall, dan F1-Score dibandingkan dengan Percobaan 1 yang hanya menggunakan Resize dan Grayscale. Jika histogram equalization berhasil membuat tekstur cacat lebih kontras, maka fitur GLCM seperti contrast dan dissimilarity berpotensi menjadi lebih diskriminatif sehingga model dapat membedakan kelas defect dan non_defect dengan lebih baik, terutama pada citra defect yang sebelumnya banyak salah diklasifikasikan sebagai non_defect (false negative).

Apabila gap antara akurasi training dan testing pada Random Forest masih besar, hal ini tetap mengindikasikan overfitting yang belum teratasi oleh preprocessing tambahan, sehingga perlu dipertimbangkan penyesuaian parameter model atau eksplorasi preprocessing lain pada percobaan selanjutnya.